# ME5.1: Amplitude Damping

## Objectives
- Apply amplitude-damping channel with time-dependent $p(t)$.
- Track relaxation: excited-state probability vs $t$.

## Setup
```python
import numpy as np, matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Kraus
```


## Theory Snapshot
- AD models $T_1$ relaxation: $P_1(t)=P_1(0)(1−p(t))$.
- Kraus: $E_0=\left[\begin{matrix}1&0\\0&\sqrt{1-p}\end{matrix}\right],\;
E_1=\left[\begin{matrix}0&\sqrt{p}\\0&0\end{matrix}\right]$

## Experiment


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Kraus

# --- Physical parameters ---
T1 = 30.0        # relaxation time (in nanoseconds)
dt = 0.5         # timestep per channel application (same units as T1)
t_max = 5*T1     # simulate up to 5 * T1

# --- Convert to per-step damping probability ---
gamma = 1.0 - np.exp(-dt / T1)  # amplitude-damping prob per step

# Kraus operators for amplitude damping with prob 'gamma'
E0 = np.array([[1, 0],[0, np.sqrt(1-gamma)]], dtype=complex)
E1 = np.array([[0, np.sqrt(gamma)],[0, 0]], dtype=complex)
AD = Kraus([E0, E1])

# --- Build and run circuits for each time point ---
sim = AerSimulator(method="density_matrix")
shots = 4096
times = np.arange(0.0, t_max + 1e-12, dt)
steps = range(len(times))

circs = []
for k in steps:
    qc = QuantumCircuit(1,1)
    qc.x(0)                     # start in |1>
    for _ in range(k): qc.append(AD, [0])   # apply channel k times
    qc.measure(0,0)
    circs.append(qc)

tqcs = transpile(circs, sim)
res = sim.run(tqcs, shots=shots).result()
p1_sim = [res.get_counts(i).get('1', 0)/shots for i in steps]

# --- Theory curves ---
p1_theory = np.exp(-times / T1)                 # exact e^{-t/T1}
p1_step_theory = (1 - gamma) ** np.arange(len(times))  # = e^{-k*dt/T1}

# --- Plot ---
plt.figure(figsize=(7,4))
plt.plot(times, p1_sim, marker='o', linestyle='', label='Simulated P(1)')
plt.plot(times, p1_theory, linestyle='--', label='Theory $e^{-t/T_1}$')
plt.plot(times, p1_step_theory, alpha=0.6, label='Discrete $(1-\\gamma)^k$')
plt.xlabel(f'Time (nanoseconds)'); plt.ylabel('Excited-state probability P(1)')
plt.title('Amplitude Damping (T1 relaxation) from $|1\\rangle$')
plt.legend(); plt.tight_layout(); plt.show()

## Results & Discussion

### Results
- $∣1\rangle$ population decays monotonically with $p(t)$.
- For representative $p, P_1\approx 1−p$ (within sampling error).

### Discussion
- Trend consistent with $T_1$-like exponential when $p(t)=1−e^{−t/T_1}$.
- Shot noise modest; choice of $p(t)$ governs decay rate.